In [1]:
import torch
import cv2
import os
import numpy as np
import pandas as pd

from pathlib import Path
from PIL import Image, ImageDraw
from tqdm import tqdm
from mmpose.apis import init_model, inference_topdown

C:\Users\sol77\miniconda3\envs\cattle_all\lib\site-packages\mmengine\utils\package_utils.py:48: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound, get_distribution


In [2]:
def _isArrayLike(obj):

    return not type(obj) == str and hasattr(obj, '__iter__') and hasattr(obj, '__len__')

In [3]:
class Databuilder:
    def __init__(
        self, img_file, save_root, width, height,
        load_model=True
    ):
        self.img = img_file
        self.save_root = Path(save_root)
        self.crop_w = int(width)
        self.crop_h = int(height)

        self.crop_save = self.save_root / "annotation" / "crop"
        self.keypoints_save = self.save_root / "annotation" / "keypoints"
        self.img_save = self.save_root / "crop_img"
        self.keypoints_vis = self.save_root / "keypoint_img"

        for folder in (
            self.crop_save,
            self.keypoints_save,
            self.img_save,
            self.keypoints_vis
        ):
            folder.mkdir(parents=True, exist_ok=True)

        self.model = None
        self.pose_model = None

        if load_model:
            self.load_yolo()

    def load_yolo(self):
        print("Loading YOLOv5...")

        self.model = torch.hub.load(
            r"C:\Users\sol77\Desktop\yolov5",
            "yolov5x",
            source="local"
        )

        self.model.conf = 0.01
        self.model.iou = 0.45
        self.model.classes = [19]  # Cow
        self.model.multi_label = False

    def load_pose(self):
        if self.pose_model is not None:
            return

        root = Path(r"C:\Users\sol77\Desktop\mmpose")

        config = root / (
            "configs/animal_2d_keypoint/topdown_heatmap/"
            "animalpose/"
            "td-hm_hrnet-w32_8xb64-210e_animalpose-256x256.py"
        )

        checkpoint = root / "checkpoints" / "animalpose.pth"

        for path in (config, checkpoint):
            if not path.is_file():
                raise FileNotFoundError(f"Missing: {path}")

        print("Loading MMPose AnimalPose...")

        self.pose_model = init_model(
            str(config),
            str(checkpoint),
            device="cpu"
        )

    def crop_images(self, img_dir, out_dir):
        if self.model is None:
            self.load_yolo()

        img_dir = Path(img_dir)
        out_dir.mkdir(parents=True, exist_ok=True)

        img_files = sorted(
            p for p in img_dir.iterdir()
            if p.suffix.lower() in (".jpg", ".jpeg", ".png")
        )

        annotations = []

        for img_path in tqdm(img_files, desc="YOLO detection"):
            output_path = out_dir / img_path.name

            if output_path.exists():
                continue

            results = self.model(str(img_path))
            detections = results.pandas().xyxy[0]

            if detections.empty:
                print(f"No cow detected: {img_path.name}")
                continue

            # Select highest-confidence cow
            bb = detections.loc[
                detections["confidence"].idxmax()
            ]

            with Image.open(img_path) as im:
                im = im.convert("RGB")
                width, height = im.size

                center_x = (bb["xmin"] + bb["xmax"]) / 2
                center_y = (bb["ymin"] + bb["ymax"]) / 2

                x = int(np.clip(
                    center_x - self.crop_w / 2,
                    0, max(0, width - self.crop_w)
                ))

                y = int(np.clip(
                    center_y - self.crop_h / 2,
                    0, max(0, height - self.crop_h)
                ))

                cropped = im.crop((
                    x, y,
                    x + self.crop_w,
                    y + self.crop_h
                ))

                cropped.save(output_path)

            annotations.append({
                "image": img_path.name,
                "xmin": float(bb["xmin"]),
                "ymin": float(bb["ymin"]),
                "xmax": float(bb["xmax"]),
                "ymax": float(bb["ymax"]),
                "confidence": float(bb["confidence"]),
                "xcrop": x,
                "ycrop": y
            })

        if annotations:
            df = pd.DataFrame(annotations)
            name = img_dir.name

            df.to_csv(
                self.crop_save / f"{name}.csv",
                index=False
            )
            df.to_hdf(
                self.crop_save / f"{name}.h5",
                key="df_with_missing",
                mode="w"
            )

    def run_pose(self, crop_dir, pose_vis=True):
        self.load_pose()

        crop_dir = Path(crop_dir)
        vis_dir = self.keypoints_vis / crop_dir.name
        vis_dir.mkdir(parents=True, exist_ok=True)

        images = sorted(
            p for p in crop_dir.iterdir()
            if p.suffix.lower() in (".jpg", ".jpeg", ".png")
        )

        records = []

        for img_path in tqdm(images, desc="Pose estimation"):
            with Image.open(img_path) as im:
                width, height = im.size

            bbox = np.array(
                [[0, 0, width, height]],
                dtype=np.float32
            )

            results = inference_topdown(
                self.pose_model,
                str(img_path),
                bboxes=bbox,
                bbox_format="xyxy"
            )

            if not results:
                continue

            pred = results[0].pred_instances
            keypoints = pred.keypoints[0]
            scores = pred.keypoint_scores[0]

            for i, ((x, y), score) in enumerate(
                zip(keypoints, scores)
            ):
                records.append({
                    "image": img_path.name,
                    "keypoint": i,
                    "x": float(x),
                    "y": float(y),
                    "score": float(score)
                })

            if pose_vis:
                image = Image.open(img_path).convert("RGB")
                draw = ImageDraw.Draw(image)

                # Draw skeleton connections if metadata provides them
                links = self.pose_model.dataset_meta.get(
                    "skeleton_links", []
                )

                for a, b in links:
                    if scores[a] < 0.3 or scores[b] < 0.3:
                        continue

                    draw.line(
                        [
                            tuple(map(float, keypoints[a])),
                            tuple(map(float, keypoints[b]))
                        ],
                        fill="lime",
                        width=3
                    )

                # Draw keypoints
                for i, ((x, y), score) in enumerate(
                    zip(keypoints, scores)
                ):
                    if score < 0.3:
                        continue

                    x, y = float(x), float(y)
                    r = 5

                    draw.ellipse(
                        (x-r, y-r, x+r, y+r),
                        fill="red"
                    )

                    draw.text(
                        (x+7, y+7),
                        str(i),
                        fill="yellow"
                    )

                output_path = vis_dir / (
                    img_path.stem + "_pose.png"
                )
                image.save(output_path)

        if records:
            df = pd.DataFrame(records)
            name = crop_dir.name

            df.to_csv(
                self.keypoints_save / f"{name}.csv",
                index=False
            )
            df.to_hdf(
                self.keypoints_save / f"{name}.h5",
                key="keypoints",
                mode="w"
            )

            print(f"Saved keypoints for {name}")

    def pipeline(
        self, needcrop=True, pose=True, pose_vis=True
    ):
        image_dirs = (
            [self.img]
            if isinstance(self.img, (str, Path))
            else self.img
        )

        for img_dir in image_dirs:
            img_dir = Path(img_dir)

            if not img_dir.is_dir():
                raise NotADirectoryError(img_dir)

            crop_dir = self.img_save / img_dir.name

            if needcrop:
                self.crop_images(img_dir, crop_dir)

            if pose:
                if not crop_dir.is_dir():
                    print(f"No cropped images: {crop_dir}")
                    continue

                self.run_pose(crop_dir, pose_vis)


In [4]:
import cv2
import os

def video_to_frames(video_path, output_folder="test_images", target_fps=15):
    os.makedirs(output_folder, exist_ok=True)

    cap = cv2.VideoCapture(video_path)

    if not cap.isOpened():
        raise ValueError(f"Could not open video: {video_path}")

    original_fps = cap.get(cv2.CAP_PROP_FPS)

    if original_fps <= 0:
        raise ValueError("Could not determine video FPS")

    print(f"Original FPS: {original_fps}")
    print(f"Target FPS: {target_fps}")

    frame_number = 0
    saved_number = 0

    while True:
        ret, frame = cap.read()

        if not ret:
            break

        # Save frames at 15 FPS
        current_time = frame_number / original_fps
        expected_saved_frames = int(current_time * target_fps) + 1

        if saved_number < expected_saved_frames:
            frame_name = f"{saved_number:06d}.jpg"
            frame_path = os.path.join(output_folder, frame_name)

            cv2.imwrite(frame_path, frame)
            saved_number += 1

        frame_number += 1

    cap.release()

    print(f"Done! Saved {saved_number} frames at {target_fps} FPS.")

if True:
    video_path = r"C:\Users\sol77\Desktop\CattleLameness\Data\Normal\N (8).mp4"
    output_folder = r"C:\Users\sol77\Desktop\test_images"
    
    video_to_frames(video_path, output_folder, target_fps=10)

Original FPS: 25.0
Target FPS: 10
Done! Saved 55 frames at 10 FPS.


In [5]:
# img_file = r"C:\Users\sol77\Desktop\test.jpg"

# img = ["Mafi-oft-ilk-cam1_Trim"]
# #img_file_list = [os.path.join(img_file, item) for item in os.listdir(img_file)]
# img_file_list = [os.path.join(img_file, item) for item in img]
img_file_list = [r"C:\Users\sol77\Desktop\test_images"]

save_file = r"C:\Users\sol77\Desktop\outputs"

data_model = Databuilder(
    img_file_list,
    save_file,
    800,
    800,
    load_model=True
)

data_model.pipeline(needcrop=True, pose=True, pose_vis=True)

Loading YOLOv5...
YOLOv5  v7.0-570-gcb2f17fd Python-3.10.22 torch-2.0.0+cpu CPU

Fusing layers... 
YOLOv5x summary: 256 layers, 86,705,005 parameters, 0 gradients
Adding AutoShape... 


YOLO detection: 100%|██████████| 55/55 [00:00<00:00, 5520.54it/s]

Loading MMPose AnimalPose...


Loads checkpoint by local backend from path: C:\Users\sol77\Desktop\mmpose\checkpoints\animalpose.pth


Pose estimation: 100%|██████████| 55/55 [01:03<00:00,  1.15s/it]


Saved keypoints for test_images
